# 📅 2026-06-28 (일) 개발 노트 : 온보딩 전구간 완성 → 마이페이지 1단계 → 취향분석 강조## 🎯 오늘의 목표- [x] 온보딩 기능 (구글 로그인 후 추가정보 수집: 이름/성별/나이대)- [x] 마이페이지 1단계 (내 정보 카드 + 로그아웃)- [x] 취향 분석 탭 강조 (네비에서 묻히던 핵심 기능)- [x] 스팀 연동은 다음으로 (메모만)> 기준일은 체감 기준(컴퓨터 껐다 켜서 하루 경과). 시스템 날짜와 무관.

## 🛠 진행 상황 및 핵심 기록**1. 온보딩 — 모델→API→폼→DB 전구간 완성 (실측)**- 목적: **데이터 수집** (성별/나이대 분포 → 향후 B2B 인구통계). 추천보다 데이터가 1차 목적.- 필수 3개 최소화(이름/성별/나이대) → 가입 이탈 방지. 닉네임/장르/게임은 선택·나중.- 나이대=구간(10s~50s_plus), 성별=male/female/other/no_answer (PIPA 안전).- 백엔드:  - `CustomUser`(AbstractUser, db_table='users')에 gender/age_group/onboarding_completed 추가. migration 0003 OK(기존유저 null 통과).  - `OnboardingSerializer`(gender/age_group 필수검증) + `OnboardingView`(POST, IsAuthenticated, 저장+completed=True).  - `UserSerializer`에 gender/age_group/onboarding_completed 추가.  - 라우트는 users/urls.py 아니라 **config/urls.py에 직접** 등록: `api/auth/onboarding/`.- 프론트:  - api.ts에 getMe()/submitOnboarding() (DJANGO_URL+토큰 fetch). **useUserStore는 함수내 동적 import**(`await import`) — 순환참조 방지 기존 패턴.  - auth/callback: onboarding_completed false→/onboarding, true→/ 분기.  - 신규 `/onboarding/page.tsx`: 폼(이름/닉네임 선택, 성별/나이대 필수버튼, canSubmit검증).- **실측**: 신규 구글계정 로그인→폼→남성/20s 선택→저장→메인. DB확인: gender=male age_group=20s onboarding_completed=True ✅**2. 마이페이지 1단계 — `/mypage`**- getMe()로 최신 정보 조회 → 내 정보 카드(닉네임/이메일/성별/나이대) + 로그아웃.- 진입: 우상단 프로필 드롭다운(이메일/마이페이지/로그아웃). 가운데 메뉴에 안 넣음(계정≠기능 분리).- 좋아하는 장르/게임은 2단계 자리만 잡아둠("곧 추가").- 온보딩 안내문구 추가: "자세한 취향은 마이페이지에서" (마이페이지 보라 강조).**3. 취향 분석 탭 강조 (Navbar v4)**- 원래 문제: 핵심기능(49지표 슬라이더 정밀추천)이 작은 텍스트 메뉴라 묻힘.- 해결: TABS에 highlight 플래그 → 보라 배경 + Sparkles 아이콘. 홈/랭킹과 시각 구분.

## 🚨 트러블슈팅 (문제 및 해결)- **문제 1:** `/mypage` 진입 시 무한 리다이렉트 루프 (mypage→login→callback→/→mypage...). "구글로 계속하기"만 반복.  - **원인:** zustand persist **hydration 타이밍**. 첫 렌더 시 localStorage 복원 전이라 isLoggedIn이 잠깐 false → useEffect가 /login으로 튕김.  - **해결:** `hydrated` state 추가. mount 후 한 틱 뒤(`setHydrated(true)`) **복원 완료 후에만** 로그인 판단. → 루프 해소.- **문제 2:** onboarding/api.ts에서 `Cannot find name 'useUserStore'`.  - **원인:** api.ts는 상단 정적 import 안 씀. 순환참조 방지로 **함수내 동적 import** 패턴.  - **해결:** getMe/submitOnboarding 각 함수 첫 줄에 `const { useUserStore } = await import('@/store/useUserStore');`.- **문제 3:** 빌드 타입에러 연쇄 — `first_name`/`steam_id` 없음(MeResponse vs UserInfo 불일치).  - **해결:** 1차로 onboarding 페이지에서 타입 우회(`as Record<string,unknown>`, setLogin 캐스팅). 이후 **store UserInfo에 gender/age_group/onboarding_completed/first_name 정식 추가**(v6)로 근본 정리.- **문제 4:** docker `Exited(255)` 4개(db/redis/django/fastapi).  - **원인:** 이번엔 마운트 깨짐 아님 — **컴퓨터 껐다 켜서** 정상 종료된 것.  - **해결:** 그냥 `docker-compose up -d` (down 불필요). → 6개 정상. (어제의 Errno5와 구분할 것)

## 💡 인사이트 및 다음 할 일**배운 점:**- **zustand persist + Next.js = hydration 함정.** 로그인 가드는 복원 완료 후 판단해야 함. 앞으로 보호 페이지(찜목록 등) 만들 때 같은 패턴 재사용.- **온보딩 ≠ "로그인 화면 살짝 고치기".** 모델+migration+폼+분기+API 묶음 기능. 데이터수집 목적이면 필수 최소화가 핵심(이탈 방지).- **"코드 있음 ≠ 동작함"** 또 적중 — 빌드 통과해도 라우트 미생성(/onboarding 폴더 없음), DB 저장은 별도 실측으로 확인.**⚠️ 기능 확장 자제 (PRD Section 23):**- 이번 세션 흐름: 칩(2-A)→온보딩(2-B 당김)→마이페이지(PRD에 없음)→"스팀도"→"하나 더"...- 이게 **scope explosion 패턴**. 한 Phase 깔끔히 안 닫고 옆으로 번짐.- **결론(합의):** 기능 그만 쌓고 **배포+마케팅**으로. PRD Phase 2 진짜 끝 = 2-C 첫 마케팅글(디시/루리웹), "기능 더"가 아님.- 스팀 연동(Phase 2-B), DNA카드, 찜 고도화 → **유저 들어온 다음**.**다음 할 일 (우선순위):**1. **배포** (다음 큰 작업, 맑은 정신에 한 세션): 배포처 결정(Vercel 프론트 + Railway 백엔드 유력) → **pgvector Railway 지원 확인** → 환경변수 이관 → CORS 재설정 → 백업 cron 서버 적용.2. 잔여: requirements.txt에 pytest 영구추가, pydantic class Config→ConfigDict.3. 미룬 기능: 스팀 OpenID 로그인, 마이페이지 2단계(장르/게임 입력+taste_dna_json 저장), 찜하기+찜목록.4. Project B(데이터 파이프라인) = 별도 프로젝트/세션. 접점: 새 게임 적재 시 gem_percentile 전체 재계산 → A 점수 스케일 검증.**현재 위치:** PRD Phase 2-A~B 사이. 회원 기능(온보딩/마이페이지) 한 바퀴 완료. 다음은 **출시(배포→마케팅)**.

## 📌 환경 메모 (반복 참고)- **표준 기동:** `docker-compose up -d` + 별터미널 `cd frontend && npm run dev`- **Errno 5 (마운트 깨짐):** `docker-compose down && up -d` 완전재생성. 3회+면 Docker Desktop 재시작.- **컴퓨터 재부팅 후 Exited:** 정상 — 그냥 `up -d` (down 불필요). Errno5와 헷갈리지 말 것.- **옆 Opus 반복실수:** `from database import AsyncSessionLocal` (맞음). `from config.database import async_session`(틀림).- **FastAPI 라우트 순서:** 고정경로(/vibes)를 동적경로(/{app_id}) 위에.- **근본예방(나중):** 프로젝트를 WSL2 내부로 이전하면 마운트 안정화.*— Vibe 칩 + 온보딩 + 마이페이지까지. 이제 만들기 그만, 내보내기 시작.*